#  Fine-Tuning BERT for POS Tagging & Chunking
### Token Classification with Hugging Face Transformers

---

**Pipeline Flow:**
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```



Install all required libraries.

In [ ]:
!pip install datasets transformers seqeval

In [ ]:
# Install required packages
!pip install transformers datasets seqeval evaluate accelerate -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

---
## Task 1: Dataset Selection

In [ ]:
dataset = load_dataset("xtreme", "udpos.English")
print(dataset)

In [ ]:
example = dataset["train"][0]
print(example)

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names
print(label_list)


##  Task 2: Data Preprocessing



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # special tokens
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])
        else:
            labels.append(-100)  # subword tokens

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=False)

---
##  Task 3: Model Setup


In [ ]:
from transformers import AutoModelForTokenClassification

label_list = dataset["train"].features["pos_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

---
## Task 4: Training


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()


##  Task 5: Evaluation

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

In [39]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [40]:
trainer.compute_metrics = compute_metrics
trainer.evaluate()

{'eval_loss': 0.13947796821594238,
 'eval_precision': 0.9524498374717064,
 'eval_recall': 0.9527416966003303,
 'eval_f1': 0.9525957446808511,
 'eval_runtime': 14.9892,
 'eval_samples_per_second': 265.125,
 'eval_steps_per_second': 16.612,
 'epoch': 2.0}

---
## Task 6: Inference


In [ ]:
from transformers import pipeline
nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

In [41]:
sentence = "Hari works at Google in New York"
outputs = nlp(sentence)

for token in outputs:
    print(token['word'], "→", token['entity'])

hari → PROPN
works → VERB
at → ADP
google → PROPN
in → ADP
new → PROPN
york → PROPN


---
##  Task 7: Comparison — POS Tagging vs Chunking
POS Tagging:

Labels each word (NOUN, VERB, etc.)
Focus on grammar
Easier task

Chunking:

Groups words into phrases (NP, VP)
Focus on structure
More complex

---
##  Task 8: Report
Differences:

POS → word-level tagging
Chunking → phrase-level grouping

Challenges Faced:

Token-label alignment
Handling subwords
Understanding -100 masking

Observations:

BERT performs well on contextual tagging
Transformer models outperform traditional NLP